# Final experiment: two signals, four combination rules
Run on an allocated BYU Slurm compute node for full panel preparation/backtesting. This notebook contains the implementation; it does not import the old repository. sf-quant and its licensed risk data must be available for the initial export. Once exported, the panel and dated risk snapshots support offline replay. Half-lives are 63, 252, and 1,260 trading sessions; these are exponential decay rates, not truncated windows.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
DATA = Path('dataset')
assert DATA.exists(), 'Run from the submission directory'
plt.rcParams.update({'figure.figsize': (11, 5), 'axes.spines.top':False, 'axes.spines.right':False})
import sys, types
pkg=types.ModuleType('BOTQ_2026');pkg.__path__=[];sys.modules['BOTQ_2026']=pkg
def register(name, source):
    module=types.ModuleType('BOTQ_2026.'+name)
    sys.modules[module.__name__]=module
    setattr(pkg,name,module)
    exec(compile(source, '<notebook:'+name+'>', 'exec'), module.__dict__)
    return module


## Implementation: combination

In [2]:
combination = register('combination', r'''
"""Leakage-controlled, exponentially weighted ridge signal combination.

Read BYU_ORC_AGENTS.md before running large panels on the cluster.
"""

from __future__ import annotations

from dataclasses import dataclass
from datetime import date

import numpy as np
import polars as pl


SIGNALS = ("dmd", "sharpe_transformer", "inflation_beta")


@dataclass(frozen=True)
class RidgeFit:
    as_of: date
    half_life_days: float
    penalty: float
    intercept: float
    coefficients: np.ndarray
    means: np.ndarray
    scales: np.ndarray
    n_observations: int

    def predict(self, signal_matrix: np.ndarray) -> np.ndarray:
        x = np.asarray(signal_matrix, dtype=float)
        if x.ndim != 2 or x.shape[1] != self.coefficients.size:
            raise ValueError(f"Expected matrix with {self.coefficients.size} signal columns")
        return self.intercept + ((x - self.means) / self.scales) @ self.coefficients


def fit_ew_ridge(
    observations: pl.DataFrame,
    *,
    as_of: date,
    half_life_days: float,
    penalty: float,
    standardize: bool,
) -> RidgeFit:
    """Fit only matured labels known before ``as_of``.

    The input must contain ``signal_date``, ``label_available_date``, the three
    signal columns, and ``forward_residual_return``. The label-available date
    is the date on which the forward return is known; the caller must include
    any extra publication delay if its data source requires one.

    Row weights decay by trading-session distance from the latest eligible
    signal date, avoiding different weights for weekends. The ridge penalty
    applies to standardized coefficients; the intercept is not penalized.
    """
    if half_life_days <= 0 or penalty < 0:
        raise ValueError("half-life must be positive and penalty nonnegative")
    required = {"signal_date", "label_available_date", "forward_residual_return", *SIGNALS}
    missing = required - set(observations.columns)
    if missing:
        raise ValueError(f"Missing columns: {sorted(missing)}")
    history = (
        observations.filter(pl.col("label_available_date") < as_of)
        .drop_nulls(list(SIGNALS) + ["forward_residual_return"])
        .sort("signal_date")
    )
    if history.height < 30:
        raise ValueError(f"Only {history.height} matured observations before {as_of}")
    dates = history["signal_date"].to_list()
    sessions = {day: i for i, day in enumerate(sorted(set(dates)))}
    age = np.asarray([len(sessions) - 1 - sessions[day] for day in dates], dtype=float)
    weights = np.exp2(-age / half_life_days)
    x = history.select(SIGNALS).to_numpy().astype(float)
    y = history["forward_residual_return"].to_numpy().astype(float)
    if not np.isfinite(x).all() or not np.isfinite(y).all():
        raise ValueError("Non-finite training values")
    total_weight = weights.sum()
    means = np.average(x, axis=0, weights=weights) if standardize else np.zeros(x.shape[1])
    scales = (
        np.sqrt(np.average((x - means) ** 2, axis=0, weights=weights))
        if standardize
        else np.ones(x.shape[1])
    )
    scales = np.maximum(scales, 1e-12)
    xz = (x - means) / scales
    y_mean = np.average(y, weights=weights)
    x_mean = np.average(xz, axis=0, weights=weights)
    xc = xz - x_mean
    yc = y - y_mean
    gram = (xc * weights[:, None]).T @ xc / total_weight
    rhs = (xc * weights[:, None]).T @ yc / total_weight
    coefficients = np.linalg.solve(gram + penalty * np.eye(len(SIGNALS)), rhs)
    intercept = float(y_mean - x_mean @ coefficients)
    return RidgeFit(
        as_of=as_of,
        half_life_days=half_life_days,
        penalty=penalty,
        intercept=intercept,
        coefficients=coefficients,
        means=means,
        scales=scales,
        n_observations=history.height,
    )


def daily_ridge_inputs(observations: pl.DataFrame, signals=SIGNALS):
    """Compress labeled rows to daily sufficient statistics for repeated fits."""
    target = "forward_residual_return"
    columns = (*signals, target)
    products = [(a, b) for i, a in enumerate(signals) for b in columns[i:]]
    daily = (
        observations.filter(pl.all_horizontal(pl.col(c).is_finite() for c in columns))
        .group_by("signal_date", "label_available_date")
        .agg(pl.len().alias("n"), *(pl.col(c).sum() for c in columns),
             *((pl.col(a) * pl.col(b)).sum().alias(f"{a}/{b}") for a, b in products))
        .sort("signal_date")
    )
    return daily, products, signals


def fit_daily_ridge(inputs, *, as_of: date, half_life_days: float, penalty: float, center: float = 0.0) -> RidgeFit:
    """Fit trailing standardized ridge from the same moments as the row-level fit."""
    daily, products, signals = inputs
    history = daily.filter(pl.col("label_available_date") < as_of)
    n = history["n"].to_numpy()
    if n.sum() < 30:
        raise ValueError(f"Only {n.sum()} matured observations before {as_of}")
    weights = np.exp2(-np.arange(len(n) - 1, -1, -1) / half_life_days)
    total = weights @ n
    means = weights @ history.select(*signals, "forward_residual_return").to_numpy() / total
    weighted_products = weights @ history.select(*(f"{a}/{b}" for a, b in products)).to_numpy() / total
    covariance = np.zeros((len(signals), len(signals)))
    cross = np.zeros(len(signals))
    for (a, b), moment in zip(products, weighted_products):
        i = signals.index(a)
        if b in signals:
            j = signals.index(b)
            covariance[i, j] = covariance[j, i] = moment - means[i] * means[j]
        else:
            cross[i] = moment - means[i] * means[-1]
    scales = np.maximum(np.sqrt(np.maximum(covariance.diagonal(), 0)), 1e-12)
    gram = covariance / scales[:, None] / scales[None, :]
    coefficients = np.linalg.solve(gram + penalty * np.eye(len(signals)), cross / scales + penalty * center)
    return RidgeFit(as_of, half_life_days, penalty, means[-1], coefficients,
                    means[:-1], scales, int(n.sum()))

''')

## Implementation: sparse_optimizer

In [3]:
sparse_optimizer = register('sparse_optimizer', r'''
"""Solve the sf-quant Barra risk problem without building a dense asset matrix."""

import clarabel
import numpy as np
from scipy import sparse


def solve(
    alpha, exposures, covariance, specific_variance, benchmark, beta,
    *, long_only, risk_cap,
):
    """Maximize alpha under an exact second-order cone tracking-error cap.

    Decision variables are active weights and their factor exposures. Keeping
    factor exposures explicit preserves sparsity in the stock risk block.
    """
    stocks, factors = exposures.shape
    size = stocks + factors
    eig, vectors = np.linalg.eigh(covariance)
    if eig.min() < -1e-10:
        raise ValueError("Barra factor covariance is not positive semidefinite")
    root = vectors * np.sqrt(np.maximum(eig, 0))
    equalities = np.vstack([np.ones(stocks), beta])
    a = [
        sparse.bmat([
            [sparse.csc_matrix(equalities), sparse.csc_matrix((2, factors))],
            [-sparse.csc_matrix(exposures.T), sparse.eye(factors)],
        ]),
        sparse.csc_matrix((1, size)),
        sparse.bmat([
            [sparse.csc_matrix((factors, stocks)), -sparse.csc_matrix(root.T)],
            [-sparse.diags(np.sqrt(specific_variance)), sparse.csc_matrix((stocks, factors))],
        ]),
    ]
    b = [np.zeros(2 + factors), np.r_[risk_cap, np.zeros(size)]]
    cones = [clarabel.ZeroConeT(2 + factors), clarabel.SecondOrderConeT(1 + size)]
    if long_only:
        a.append(sparse.hstack([-sparse.eye(stocks), sparse.csc_matrix((stocks, factors))]))
        b.append(benchmark)
        cones.append(clarabel.NonnegativeConeT(stocks))
    settings = clarabel.DefaultSettings()
    settings.verbose = False
    settings.max_threads = 1
    result = clarabel.DefaultSolver(
        sparse.csc_matrix((size, size)), np.r_[-alpha, np.zeros(factors)],
        sparse.vstack(a, format="csc"), np.concatenate(b), cones, settings,
    ).solve()
    if result.status not in (clarabel.SolverStatus.Solved, clarabel.SolverStatus.AlmostSolved):
        raise ValueError(f"Sparse Barra optimizer failed: {result.status}")
    weights = np.asarray(result.x[:stocks])
    risk = np.linalg.norm(np.r_[root.T @ (exposures.T @ weights), np.sqrt(specific_variance) * weights])
    return weights, float(risk)

''')

## Implementation: backtest

In [4]:
backtest = register('backtest', r'''
"""Daily next-close portfolio accounting and plots for BOTQ_2026.

Targets are set after close on date t and held from the next close, so weights
on t multiply the close-to-close return recorded on t+1. Trading costs are
zero by the user's instruction; reported performance is gross.
"""

from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import polars as pl


def active_sharpe(returns: np.ndarray, annualization: int = 252) -> float:
    """Annualized Sharpe of daily active returns, with zero risk-free spread."""
    r = np.asarray(returns, dtype=float)
    r = r[np.isfinite(r)]
    if len(r) < 2 or np.std(r, ddof=1) == 0:
        return float("nan")
    return float(np.sqrt(annualization) * r.mean() / r.std(ddof=1))


def plot_results(returns: pl.DataFrame, coefficients: pl.DataFrame, output: Path) -> None:
    """Save total wealth vs Russell, active wealth, and coefficients by variant."""
    output.mkdir(parents=True, exist_ok=True)
    data = returns.sort("date").to_pandas()
    if (data[["portfolio_return", "benchmark_return"]] <= -1).any().any():
        raise ValueError("Log returns require positive daily gross returns")
    fig, ax = plt.subplots(figsize=(12, 6))
    for name, group in data.groupby("portfolio"):
        wealth = np.log1p(group["portfolio_return"]).cumsum()
        ax.plot(group["date"], wealth, label=f"{name} total")
    bench = data.groupby("date")["benchmark_return"].first()
    ax.plot(bench.index, np.log1p(bench).cumsum(), color="black", label="Russell 3000")
    ax.set(title="Portfolio and benchmark performance", ylabel="Cumulative log return")
    ax.grid(alpha=.25); ax.legend(); fig.tight_layout()
    fig.savefig(output / "total_vs_russell.png", dpi=160); plt.close(fig)

    fig, ax = plt.subplots(figsize=(12, 5))
    for name, group in data.groupby("portfolio"):
        active_wealth = (np.log1p(group["portfolio_return"]) - np.log1p(group["benchmark_return"])).cumsum()
        ax.plot(group["date"], active_wealth, label=name)
    ax.axhline(0, color="black", linewidth=.8)
    ax.set(title="Performance relative to the Russell 3000", ylabel="Cumulative excess log return")
    ax.grid(alpha=.25); ax.legend(); fig.tight_layout()
    fig.savefig(output / "active_performance.png", dpi=160); plt.close(fig)

    if coefficients.is_empty():
        return
    c = coefficients.sort("date").to_pandas()
    count = c["portfolio"].nunique()
    fig, axes = plt.subplots(count, 1, figsize=(12, 4 * count), sharex=True, squeeze=False)
    for ax, (name, group) in zip(axes[:, 0], c.groupby("portfolio")):
        for signal in ("dmd", "sharpe_transformer", "inflation_beta"):
            if signal in group:
                ax.plot(group["date"], group[signal], label=signal)
        ax.axhline(0, color="black", linewidth=.8)
        ax.set_title(f"Ridge coefficients: {name}"); ax.grid(alpha=.25); ax.legend()
    fig.tight_layout(); fig.savefig(output / "ridge_coefficients.png", dpi=160); plt.close(fig)

''')

## Implementation: portfolio

In [5]:
portfolio = register('portfolio', r'''
"""One combined-alpha optimization in active weight space.

The constraints and risk units are inherited from sf-quant. Read the
repository's BYU_ORC_AGENTS.md before running this on ORC.
"""

from __future__ import annotations

from datetime import date
from functools import lru_cache

import numpy as np
import polars as pl


TARGET_TRACKING_ERROR = 0.025


@lru_cache(maxsize=800)
def _daily_risk(day: date, ids: tuple[str, ...]):
    """Reuse dated Barra inputs across portfolios."""
    from sf_quant.data import construct_factor_model_components

    return construct_factor_model_components(day, list(ids))


def optimize_day(
    day: date,
    panel: pl.DataFrame,
    *,
    long_only: bool,
    sparse_solver: bool = False,
) -> pl.DataFrame:
    """Optimize total and active weights for one dated Russell cross-section.

    ``panel`` needs unique ``barrid`` values and the columns ``combined_alpha``,
    ``benchmark_weight``, and ``predicted_beta``. Incomplete risk or signal
    inputs fail visibly rather than silently changing the benchmark universe.
    The beta target is measured on active weights relative to the benchmark.
    """
    from sf_quant.optimizer import (
        LongOnly,
        ZeroBeta,
        ZeroInvestment,
        active_mve_optimizer,
    )

    required = {"barrid", "combined_alpha", "benchmark_weight", "predicted_beta"}
    if missing := required - set(panel.columns):
        raise ValueError(f"{day}: missing columns {sorted(missing)}")
    if panel["barrid"].n_unique() != panel.height:
        raise ValueError(f"{day}: duplicate barrid values")
    if any(panel[c].null_count() for c in required):
        raise ValueError(f"{day}: missing optimization inputs")
    ids = panel["barrid"].to_list()
    alpha = panel["combined_alpha"].to_numpy().astype(float)
    benchmark = panel["benchmark_weight"].to_numpy().astype(float)
    beta = panel["predicted_beta"].to_numpy().astype(float)
    if not all(np.isfinite(x).all() for x in (alpha, benchmark, beta)):
        raise ValueError(f"{day}: non-finite optimization inputs")
    if not np.isclose(benchmark.sum(), 1.0, atol=1e-6):
        raise ValueError(f"{day}: benchmark weights sum to {benchmark.sum():.8f}")
    exposures, factor_covariance, specific_variance = _daily_risk(day, tuple(ids))
    constraints = [ZeroInvestment(), ZeroBeta()]
    if long_only:
        constraints.append(LongOnly())
    # A positive scaling preserves the optimizer and keeps tiny forecasts
    # from setting the numerical scale of the conic objective.
    scale = max(float(np.max(np.abs(alpha))), np.finfo(float).tiny)
    from cvxpy.error import SolverError

    try:
        if sparse_solver:
            from BOTQ_2026.sparse_optimizer import solve

            active, predicted_te = solve(
                alpha / scale, exposures, factor_covariance, specific_variance,
                benchmark, beta, long_only=long_only,
                risk_cap=TARGET_TRACKING_ERROR,
            )
        else:
            result = active_mve_optimizer(
                ids=ids,
                alphas=alpha / scale,
                factor_exposures=exposures,
                factor_covariance=factor_covariance,
                specific_risk=specific_variance,
                constraints=constraints,
                benchmark_weights=benchmark,
                target_active_risk=TARGET_TRACKING_ERROR,
                betas=beta,
            )
            active = result["active_weight"].to_numpy()
            predicted_te = float(result["active_risk"][0])
    except SolverError:
        active, predicted_te = _solve_soc(
            alpha / scale, exposures, factor_covariance, specific_variance,
            constraints, benchmark, beta,
        )

    total = active + benchmark
    if abs(active.sum()) > 1e-5 or abs(beta @ active) > 1e-5:
        raise ValueError(f"{day}: investment or active-beta constraint violated")
    if long_only and total.min() < -1e-6:
        raise ValueError(f"{day}: long-only constraint violated")
    if predicted_te > TARGET_TRACKING_ERROR + 1e-5:
        raise ValueError(f"{day}: tracking-error cap violated")
    return pl.DataFrame(
        {
            "date": [day] * len(ids),
            "barrid": ids,
            "benchmark_weight": benchmark,
            "active_weight": active,
            "total_weight": total,
            "predicted_beta": beta,
            "combined_alpha": alpha,
            "predicted_tracking_error": [predicted_te] * len(ids),
            "portfolio": ["long_only" if long_only else "active"] * len(ids),
        }
    )


def _solve_soc(
    alpha, exposures, covariance, specific_variance, constraints, benchmark, beta,
    *, risk_cap=TARGET_TRACKING_ERROR,
):
    """Maximize a linear objective with the Barra risk bound expressed as a norm."""
    import cvxpy as cp

    eig, vectors = np.linalg.eigh(covariance)
    root = vectors * np.sqrt(np.maximum(eig, 0))
    active = cp.Variable(len(alpha))
    factor = cp.Variable(exposures.shape[1])
    risk_vector = cp.hstack([root.T @ factor, cp.multiply(np.sqrt(specific_variance), active)])
    limits = [factor == exposures.T @ active, cp.norm(risk_vector) <= risk_cap]
    limits.extend(c(active, benchmark_weights=benchmark, betas=beta) for c in constraints)
    problem = cp.Problem(cp.Maximize(alpha @ active), limits)
    try:
        problem.solve(solver="CLARABEL", max_threads=1)
    except cp.error.SolverError:
        problem.solve(solver="SCS", eps=1e-7, max_iters=30000)
    if problem.status not in (cp.OPTIMAL, cp.OPTIMAL_INACCURATE) or active.value is None:
        raise ValueError(f"Fallback portfolio optimization failed: {problem.status}")
    weights = active.value
    risk = np.linalg.norm(np.r_[root.T @ (exposures.T @ weights), np.sqrt(specific_variance) * weights])
    return weights, float(risk)

''')

## Implementation: dmd_attribution

In [6]:
dmd_attribution = register('dmd_attribution', r'''
"""Attribute DMD's stock-level forecast and forward residual payoff to styles."""
import argparse
from datetime import date
from pathlib import Path

import numpy as np
import polars as pl

CONTROLS = {
    "barra_momentum": "USSLOWL_MOMENTUM",
    "barra_reversal": "USSLOWL_LTREVRSL",
    "sf_momentum": "momentum",
    "sf_reversal": "reversal",
}
SPECS = {
    **{name: (name,) for name in CONTROLS},
    "momentum": ("barra_momentum", "sf_momentum"),
    "reversal": ("barra_reversal", "sf_reversal"),
    "barra": ("barra_momentum", "barra_reversal"),
    "sf_quant": ("sf_momentum", "sf_reversal"),
    "all": tuple(CONTROLS),
}


def join_style_controls(panel):
    """Join same-signal-date style controls for a single calendar year."""
    import sf_quant.data as sfd

    start, end = panel["signal_date"].min(), panel["signal_date"].max()
    exposures = sfd.load_exposures(start, end, False, [CONTROLS["barra_momentum"], CONTROLS["barra_reversal"]])
    signals = sfd.load_scores(start, end, names=[CONTROLS["sf_momentum"], CONTROLS["sf_reversal"]])
    for source, names, value in (
        (exposures, ("barra_momentum", "barra_reversal"), None),
        (signals, ("sf_momentum", "sf_reversal"), "score"),
    ):
        if value:
            source = source.group_by("date", "barrid").agg(*(
                pl.col(value).filter(pl.col("signal_name") == CONTROLS[name]).first().alias(name)
                for name in names
            ))
        else:
            source = source.rename({CONTROLS[name]: name for name in names})
        panel = panel.join(source.rename({"date": "signal_date"}),
                             on=["signal_date", "barrid"], how="left", validate="m:1")
    return panel


def dmd_style_scores(day):
    """Return residual DMD and raw DMD on identical coverage."""
    dmd = day["dmd"].to_numpy()
    styles = day.select(*CONTROLS).to_numpy()
    matched = np.isfinite(dmd) & np.isfinite(styles).all(axis=1)
    if matched.sum() < 30:
        raise ValueError(f"Insufficient DMD style coverage on {day['signal_date'][0]}")
    dmd_matched = dmd[matched] - dmd[matched].mean()
    x = styles[matched] - styles[matched].mean(axis=0)
    residual = np.zeros(len(dmd))
    residual[matched] = dmd_matched - x @ np.linalg.lstsq(x, dmd_matched, rcond=None)[0]
    raw = np.zeros(len(dmd))
    raw[matched] = dmd_matched
    return residual, raw


def daily_attribution(dmd, y, styles):
    """OLS projection within one signal date. Payoff is exactly additive."""
    dmd = dmd - dmd.mean()
    y = y - y.mean()
    styles = styles - styles.mean(axis=0)
    ss = dmd @ dmd
    gross = np.abs(dmd).sum()
    if ss <= 0:
        return []
    result = []
    raw = (dmd @ y) / gross
    for spec, controls in SPECS.items():
        cols = [list(CONTROLS).index(c) for c in controls]
        x = styles[:, cols]
        fitted = x @ np.linalg.lstsq(x, dmd, rcond=None)[0]
        residual = dmd - fitted
        residual_ss = residual @ residual
        result.append(dict(
            controls=spec, score_r2=1 - residual_ss / ss,
            raw_payoff=raw, explained_payoff=(fitted @ y) / gross,
            remaining_payoff=(residual @ y) / gross,
            raw_slope=dmd @ y / ss,
            adjusted_slope=residual @ y / residual_ss if residual_ss > 1e-12 else None,
            score_correlation=(dmd @ x[:, 0]) / np.sqrt(ss * (x[:, 0] @ x[:, 0]))
            if len(cols) == 1 and np.any(x[:, 0]) else None,
        ))
    return result


def main():
    import sf_quant.data as sfd
    from BOTQ_2026.walk_forward import build_panel

    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("cache", type=Path)
    parser.add_argument("output", type=Path)
    args = parser.parse_args()
    sfd.env(root=str(Path.home()), database="production")
    assets = pl.scan_parquet(args.cache / "assets.parquet").filter(
        pl.col("date").is_between(date(2009, 12, 1), date(2024, 12, 31))
    ).select("date", "barrid", "in_universe", "market_cap", "return", "predicted_beta").collect()
    forecasts = pl.scan_parquet(args.cache / "forecasts-through-2024.parquet").select(
        "signal_date", "barrid", "dmd").collect()
    panel = build_panel(assets, forecasts, pl.read_parquet(args.cache / "exit_return_audit.parquet")).filter(
        pl.col("date") >= date(2010, 1, 1)
    ).select("signal_date", "label_available_date", "barrid", "dmd", "forward_residual_return")
    rows, coverage = [], []
    for year in range(2009, 2025):
        subset = panel.filter(pl.col("signal_date").dt.year() == year)
        start, end = subset["signal_date"].min(), subset["signal_date"].max()
        if start is None:
            continue
        subset = join_style_controls(subset)
        for group in subset.partition_by("signal_date", maintain_order=True):
            dt = group["signal_date"][0]
            eligible = group.filter(pl.col("dmd").is_finite() & pl.col("forward_residual_return").is_finite())
            matched = eligible.filter(pl.all_horizontal(pl.col(c).is_finite() for c in CONTROLS))
            coverage.append(dict(signal_date=dt, eligible=eligible.height, matched=matched.height,
                                 **{name: eligible[name].is_finite().sum() for name in CONTROLS}))
            if matched.height >= 30:
                rows.extend(dict(signal_date=dt, stocks=matched.height, **row)
                            for row in daily_attribution(matched["dmd"].to_numpy(),
                                                        matched["forward_residual_return"].to_numpy(),
                                                        matched.select(*CONTROLS).to_numpy()))
        print(f"{year}: cumulative regression rows {len(rows)}", flush=True)
    args.output.mkdir(parents=True, exist_ok=True)
    pl.DataFrame(coverage).sort("signal_date").write_csv(args.output / "coverage.csv")
    if not rows:
        raise ValueError("No overlapping DMD and style score dates; inspect coverage.csv")
    daily = pl.DataFrame(rows).sort("signal_date", "controls")
    daily.write_parquet(args.output / "daily.parquet")
    summary = daily.group_by("controls").agg(
        pl.len().alias("sessions"), pl.col("stocks").mean(),
        *(pl.col(c).mean() for c in ("score_r2", "score_correlation", "raw_payoff",
                                     "explained_payoff", "remaining_payoff", "raw_slope", "adjusted_slope")),
    ).with_columns((pl.col("explained_payoff") / pl.col("raw_payoff")).alias("payoff_share_explained"))
    summary.sort("controls").write_csv(args.output / "summary.csv")

    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    both = daily.filter(pl.col("controls") == "all")
    fig, ax = plt.subplots(figsize=(10, 5))
    for col, label in (("raw_payoff", "Raw DMD"), ("explained_payoff", "Style component"),
                       ("remaining_payoff", "DMD after controls")):
        ax.plot(both["signal_date"].to_list(), both[col].cum_sum().to_numpy(), label=label)
    ax.set(ylabel="Cumulative residual payoff per unit of raw DMD gross",
           xlabel="Signal date", title="DMD payoff attribution: Barra and sf-quant")
    ax.legend()
    fig.tight_layout()
    fig.savefig(args.output / "dmd_payoff_attribution.png", dpi=150)


if __name__ == "__main__":
    main()

''')

## Implementation: model_candidates

In [7]:
model_candidates = register('model_candidates', r'''
"""Fixed coefficient rules for the walk-forward ensemble."""

from datetime import date
from itertools import product

import numpy as np

from BOTQ_2026.combination import SIGNALS, fit_daily_ridge

HALF_LIVES = (63, 252, 1260)
PENALTIES = (1.0, 0.05, 0.0025)


def candidate_alphas(inputs, as_of: date, scores: np.ndarray):
    """Yield each candidate's alpha and its coefficients on standardized inputs."""
    for half_life, penalty in product(HALF_LIVES, PENALTIES):
        fit = fit_daily_ridge(inputs, as_of=as_of, half_life_days=half_life, penalty=penalty)
        standardized = (scores - fit.means) / fit.scales
        standardized = np.where(np.isfinite(standardized), standardized, 0.0)
        if (half_life, penalty) == (HALF_LIVES[0], PENALTIES[0]):
            # Keep the same 126-session standardization as the earlier equal-weight run.
            equal_fit = fit_daily_ridge(inputs, as_of=as_of, half_life_days=126, penalty=1.0)
            equal_scores = (scores - equal_fit.means) / equal_fit.scales
            equal_scores = np.where(np.isfinite(equal_scores), equal_scores, 0.0)
            coefficients = np.full(len(SIGNALS), 1 / len(SIGNALS))
            yield "equal_weight", equal_scores @ coefficients, coefficients
        name = f"ridge_{half_life}_p{penalty:g}"
        yield name, standardized @ fit.coefficients, fit.coefficients


CENTERED_SPECS = ((63, 1.0), (1260, 0.0025))
CENTER_WARMUP = 252


def centered_fits(inputs, dates):
    """Fit in chronological order, initializing each series with ordinary ridge."""
    centers = dict.fromkeys(CENTERED_SPECS, 0.0)
    result = {}
    for index, day in enumerate(dates):
        result[day] = {}
        for half_life, penalty in CENTERED_SPECS:
            key = (half_life, penalty)
            center = centers[key]
            fit = fit_daily_ridge(inputs, as_of=day, half_life_days=half_life,
                                  penalty=penalty, center=center)
            if index < CENTER_WARMUP:
                centers[key] = float(fit.coefficients.max())
            result[day][f"centered_{half_life}_p{penalty:g}"] = (fit, center)
    return result


def portfolio_posteriors(returns, execution_dates, names=None):
    """Update uniform model odds with each candidate's active-return evidence.

    Compare a Gaussian positive-alpha model (1% annual active mean) to a
    zero-mean reference, both with 2.5% annual volatility. The daily log
    likelihood ratio is linear in the realized return. Use the same reference
    for all candidates, and normalize relative evidence across candidates.
    """
    import polars as pl
    from scipy.special import softmax

    if names is None:
        names = ["equal_weight", *(
            f"ridge_{half_life}_p{penalty:g}" for half_life, penalty in product(HALF_LIVES, PENALTIES)
        )]
    dates = np.asarray(execution_dates, dtype="datetime64[D]")
    posteriors = {}
    for variant in ("active", "long_only"):
        columns = [f"{name}_{variant}" for name in names]
        history = returns.filter(pl.col("portfolio").is_in(columns)).pivot(
            on="portfolio", index="date", values="active_return",
        ).sort("date")
        if set(history.columns) != {"date", *columns} or history.height != len(dates):
            raise ValueError(f"Incomplete candidate return history for {variant}")
        realized = history.select(columns).to_numpy()
        if not np.isfinite(realized).all():
            raise ValueError(f"Non-finite candidate returns for {variant}")
        # A return ending on the execution date is not known before that close.
        known = np.searchsorted(history["date"].to_numpy(), dates, side="left")
        cumulative = np.vstack([np.zeros(len(names)), realized.cumsum(axis=0)])
        posterior = softmax((0.01 / 0.025**2) * cumulative[known], axis=1)
        posteriors[variant] = dict(zip(execution_dates, posterior))
    return names, posteriors

''')

## Implementation: walk_forward

In [8]:
walk_forward = register('walk_forward', r'''
"""Backtest fixed ridge and equal-weight combinations of dated forecasts.

Forecasts must have one row per signal_date/barrid. The execution date is
the next market session, and the holding period ends one session later.
Returns for exits must be supplied in the asset history, including delistings.
"""
from __future__ import annotations

import argparse
from concurrent.futures import ThreadPoolExecutor
from datetime import date
from pathlib import Path

import numpy as np
import polars as pl

from BOTQ_2026.backtest import active_sharpe, plot_results
from BOTQ_2026.combination import SIGNALS, daily_ridge_inputs, fit_daily_ridge
from BOTQ_2026.portfolio import optimize_day
from BOTQ_2026.dmd_attribution import dmd_style_scores, join_style_controls
from BOTQ_2026.model_candidates import candidate_alphas, centered_fits, portfolio_posteriors


def build_panel(assets: pl.DataFrame, forecasts: pl.DataFrame, exits: pl.DataFrame | None = None) -> pl.DataFrame:
    """Align dates on a common exchange calendar, including for stocks that exit."""
    calendar = assets.select("date").unique().sort("date").with_columns(
        pl.col("date").shift(1).alias("signal_date"),
        pl.col("date").shift(-1).alias("label_available_date"),
    )
    members = assets.filter(pl.col("in_universe")).with_columns(
        (pl.col("market_cap") / pl.col("market_cap").sum().over("date")).alias("benchmark_weight")
    )
    # sf-quant stores daily returns in percentage points. CRSP fallbacks are decimals.
    realized = assets.select(
        pl.col("date").alias("label_available_date"),
        "barrid",
        (pl.col("return") / 100).alias("holding_return"),
    )
    if exits is not None:
        realized = realized.join(
            exits.select("label_available_date", "barrid", "delret", "crsp_return"),
            on=["label_available_date", "barrid"], how="full", coalesce=True, validate="1:1",
        ).with_columns(
            pl.coalesce("holding_return", "delret", "crsp_return").alias("holding_return")
        ).drop("delret", "crsp_return")
    panel = (
        members.join(calendar, on="date", validate="m:1")
        .drop_nulls(["signal_date", "label_available_date"])
        .join(forecasts, on=["signal_date", "barrid"], how="left", validate="m:1")
        .join(realized, on=["label_available_date", "barrid"], how="left", validate="m:1")
        .with_columns(
            pl.col("holding_return").is_null().sum().over("date").alias("carried_names"),
            pl.col("holding_return").fill_null(0),
        )
        .with_columns(
            (pl.col("benchmark_weight") * pl.col("holding_return")).sum().over("date").alias("benchmark_return")
        )
        .with_columns(
            (pl.col("holding_return") - pl.col("predicted_beta") * pl.col("benchmark_return"))
            .alias("forward_residual_return")
        )
    )
    return panel.sort(["date", "barrid"])


RIDGE_HALF_LIFE = 126  # Six months in trading sessions.
RIDGE_PENALTY = 0.2


def run(
    panel: pl.DataFrame,
    output: Path,
    *,
    end_date: date,
    workers: int = 1,
    sparse_solver: bool = False,
    single_signal: str | None = None,
    ridge_half_life: float = RIDGE_HALF_LIFE,
    ridge_penalty: float = RIDGE_PENALTY,
    ridge_only: bool = False,
    candidate_grid: bool = False,
    centered_grid: bool = False,
    dmd_inflation: bool = False,
    include_equal: bool = False,
    candidate_returns: pl.DataFrame | None = None,
) -> None:
    """Fit with matured labels and compare both combinations on identical dates."""
    signals = ("dmd", "inflation_beta") if dmd_inflation else SIGNALS
    if single_signal is None:
        ridge_inputs = daily_ridge_inputs(panel.select(
            "signal_date", "label_available_date", "forward_residual_return", *signals
        ), signals=signals)
    test_panel = panel.filter(
        (pl.col("date") >= date(2010, 1, 1)) & (pl.col("label_available_date") <= end_date)
    )
    if dmd_inflation and candidate_returns is None and not test_panel.is_empty():
        test_panel = pl.concat([
            join_style_controls(year) for year in
            test_panel.with_columns(pl.col("signal_date").dt.year().alias("_style_year"))
            .partition_by("_style_year", include_key=False)
        ]).sort("date", "barrid")
    test = test_panel.partition_by("date", maintain_order=True)
    if not test:
        raise ValueError("No backtest sessions in the requested date range")

    execution_dates = [day["date"][0] for day in test]
    fits = centered_fits(ridge_inputs, execution_dates) if centered_grid else None
    include_equal = include_equal or dmd_inflation
    if candidate_returns is not None:
        names = list(fits[execution_dates[0]]) if fits else None
        if names is not None and include_equal:
            names.append("equal_weight")
        model_names, posteriors = portfolio_posteriors(candidate_returns, execution_dates, names)

    def day_candidates(execution, scores):
        if fits is None:
            yield from candidate_alphas(ridge_inputs, execution, scores)
            return
        for name, (fit, _) in fits[execution].items():
            z = (scores - fit.means) / fit.scales
            yield name, np.where(np.isfinite(z), z, 0.0) @ fit.coefficients, fit.coefficients
        if include_equal:
            fit = fit_daily_ridge(ridge_inputs, as_of=execution, half_life_days=126, penalty=1.0)
            z = (scores - fit.means) / fit.scales
            z = np.where(np.isfinite(z), z, 0.0)
            weights = np.full(len(signals), 1 / len(signals))
            yield "equal_weight", z @ weights, weights
            if dmd_inflation and candidate_returns is None:
                weights = np.array([0.95, 0.05])
                yield "dmd95_inflation5", z @ weights, weights

    def evaluate_day(day):
        execution = day["date"][0]
        if candidate_returns is not None:
            candidates = {name: alpha for name, alpha, _ in
                          day_candidates(execution, day.select(signals).to_numpy())}
            matrix = np.column_stack([candidates[name] for name in model_names])
            # Standalone optimizers are scale invariant. Normalize dispersion
            # before blending so posterior weights control relative influence.
            dispersion = matrix.std(axis=0)
            matrix = np.divide(matrix, dispersion, out=np.zeros_like(matrix), where=dispersion > 0)
            alphas = {"bayesian": {variant: matrix @ weights[execution]
                                   for variant, weights in posteriors.items()}}
            coefficients = []
        elif candidate_grid or centered_grid:
            candidates = list(day_candidates(execution, day.select(signals).to_numpy()))
            alphas = {name: alpha for name, alpha, _ in candidates}
            coefficients = []
            for name, _, weights in candidates:
                row = {"date": execution, "portfolio": name, **dict(zip(signals, weights))}
                if fits is not None:
                    ridge = fits[execution].get(name)
                    row["ridge_center"] = ridge[1] if ridge else None
                coefficients.append(row)
            if dmd_inflation:
                residual, raw = dmd_style_scores(day)
                alphas["dmd_residual"] = residual
                alphas["dmd_matched_raw"] = raw
        elif single_signal is None:
            fit = fit_daily_ridge(
                ridge_inputs, as_of=execution, half_life_days=ridge_half_life,
                penalty=ridge_penalty,
            )
            scores = (day.select(signals).to_numpy() - fit.means) / fit.scales
            # Use the same inputs for both combinations. A missing forecast
            # contributes zero in standardized units.
            scores = np.where(np.isfinite(scores), scores, 0.0)
            alphas = {"ridge": scores @ fit.coefficients}
            if not ridge_only:
                alphas["equal_weight"] = scores.mean(axis=1)
            coefficients = [{"date": execution, "portfolio": "ridge",
                             **dict(zip(SIGNALS, fit.coefficients))}]
        else:
            scores = day[single_signal].to_numpy()
            alphas = {single_signal: np.where(np.isfinite(scores), scores, 0.0)}
            coefficients = []

        rows = []
        for method, alpha in alphas.items():
            for long_only in ((False, True) if single_signal is None else (False,)):
                variant = "long_only" if long_only else "active"
                portfolio = f"{method}_{variant}"
                score = alpha[variant] if candidate_returns is not None else alpha
                alpha_panel = day.with_columns(pl.Series("combined_alpha", score))
                try:
                    weights = optimize_day(
                        execution, alpha_panel, long_only=long_only, sparse_solver=sparse_solver,
                    ).join(day.select("barrid", "holding_return"), on="barrid", validate="1:1")
                except Exception as exc:
                    raise RuntimeError(f"{portfolio} failed on {execution}") from exc
                realized = weights["holding_return"].to_numpy()
                rows.append(dict(
                    date=day["label_available_date"][0], portfolio=portfolio,
                    active_return=float(weights["active_weight"].to_numpy() @ realized),
                    portfolio_return=float(weights["total_weight"].to_numpy() @ realized),
                    benchmark_return=day["benchmark_return"][0],
                    carried_names=day["carried_names"][0],
                    predicted_tracking_error=weights["predicted_tracking_error"][0],
                    active_beta=float(weights["active_weight"].to_numpy() @ weights["predicted_beta"].to_numpy()),
                ))
        return execution, rows, coefficients

    returns, coefficients = [], []
    # Solve all variants for a date together so they share the same Barra load.
    with ThreadPoolExecutor(max_workers=workers) as pool:
        for index, (execution, rows, coefficient_row) in enumerate(pool.map(evaluate_day, test), 1):
            returns.extend(rows)
            coefficients.extend(coefficient_row)
            if index % 250 == 0 or index == len(test):
                print(f"Backtest: {index}/{len(test)} sessions through {execution}", flush=True)

    output.mkdir(parents=True, exist_ok=True)
    performance = pl.DataFrame(returns)
    coefficient_history = pl.DataFrame(coefficients)
    performance.write_parquet(output / "returns.parquet")
    if candidate_returns is not None:
        pl.DataFrame([
            {"date": day, "portfolio": variant, **dict(zip(model_names, weights))}
            for variant, series in posteriors.items() for day, weights in series.items()
        ]).write_parquet(output / "model_weights.parquet")
    if not coefficient_history.is_empty():
        coefficient_history.write_parquet(output / "coefficients.parquet")
    summary = []
    for group in performance.partition_by("portfolio"):
        daily = group["active_return"].to_numpy()
        summary.append({
            "portfolio": group["portfolio"][0],
            "first_return": group["date"].min(),
            "last_return": group["date"].max(),
            "sessions": group.height,
            "portfolio_cumulative_return": float(np.prod(1 + group["portfolio_return"].to_numpy()) - 1),
            "benchmark_cumulative_return": float(np.prod(1 + group["benchmark_return"].to_numpy()) - 1),
            "active_sharpe": active_sharpe(daily),
            "realized_tracking_error": float(np.std(daily, ddof=1) * np.sqrt(252)),
            "max_predicted_tracking_error": group["predicted_tracking_error"].max(),
            "max_absolute_active_beta": group["active_beta"].abs().max(),
            "carried_name_sessions": group["carried_names"].sum(),
        })
    pl.DataFrame(summary).write_csv(output / "summary.csv")
    plot_results(performance, coefficient_history, output)


def main() -> None:
    import sf_quant.data as sfd

    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--workers", type=int, default=1)
    parser.add_argument("--sparse-solver", action="store_true")
    parser.add_argument("--single-signal", choices=SIGNALS, help="Use one raw forecast as the sole alpha")
    parser.add_argument("--assets", type=Path, required=True)
    parser.add_argument("--forecasts", type=Path, required=True)
    parser.add_argument("--out", type=Path, required=True)
    parser.add_argument("--exits", type=Path)
    parser.add_argument("--end-date", type=date.fromisoformat, required=True)
    parser.add_argument("--root", type=Path, default=Path.home())
    parser.add_argument("--ridge-half-life", type=float, default=RIDGE_HALF_LIFE)
    parser.add_argument("--ridge-penalty", type=float, default=RIDGE_PENALTY)
    parser.add_argument("--ridge-only", action="store_true")
    parser.add_argument("--candidate-grid", action="store_true")
    parser.add_argument("--candidate-returns", type=Path, help="Fit portfolio-based Bayesian model weights")
    parser.add_argument("--centered-grid", action="store_true")
    parser.add_argument("--include-equal", action="store_true")
    parser.add_argument("--dmd-inflation", action="store_true",
                        help="Use DMD and inflation beta in the centered experiment")
    args = parser.parse_args()
    if args.dmd_inflation and not args.centered_grid:
        parser.error("--dmd-inflation requires --centered-grid")
    if args.centered_grid and (args.candidate_grid or args.single_signal or args.ridge_only):
        parser.error("--centered-grid is incompatible with other standalone run modes")
    if args.include_equal and not (args.centered_grid and args.candidate_returns):
        parser.error("--include-equal requires centered grid and merged candidate returns")
    if args.candidate_returns and (args.candidate_grid or args.single_signal or args.ridge_only):
        parser.error("--candidate-returns is a separate run mode")
    if args.candidate_grid and (args.single_signal or args.ridge_only):
        parser.error("--candidate-grid cannot be combined with --single-signal or --ridge-only")
    if not np.isfinite(args.ridge_half_life) or args.ridge_half_life <= 0:
        parser.error("--ridge-half-life must be finite and positive")
    if not np.isfinite(args.ridge_penalty) or args.ridge_penalty < 0:
        parser.error("--ridge-penalty must be finite and nonnegative")
    sfd.env(root=str(args.root), database="production")
    exits = pl.read_parquet(args.exits) if args.exits else None
    assets = (
        pl.scan_parquet(args.assets)
        .filter(pl.col("date").is_between(date(2005, 1, 1), args.end_date))
        .select("date", "barrid", "in_universe", "market_cap", "return", "predicted_beta")
        .collect()
    )
    run(
        build_panel(assets, pl.read_parquet(args.forecasts), exits),
        args.out, end_date=args.end_date, workers=args.workers,
        sparse_solver=args.sparse_solver, single_signal=args.single_signal,
        ridge_half_life=args.ridge_half_life, ridge_penalty=args.ridge_penalty,
        ridge_only=args.ridge_only, candidate_grid=args.candidate_grid,
        centered_grid=args.centered_grid, dmd_inflation=args.dmd_inflation,
        include_equal=args.include_equal,
        candidate_returns=pl.read_parquet(args.candidate_returns) if args.candidate_returns else None,
    )


if __name__ == "__main__":
    main()

''')

## Implementation: dmd

In [9]:
dmd = register('dmd', r'''
"""Hankel delay-coordinate DMD directional signal adapted from DMDBacktestV3.

Use split-adjusted close prices and only observations through the signal date.
This module returns a per-stock direction; the walk-forward layer owns model
selection and delayed execution alignment.
"""

from __future__ import annotations

import numpy as np


def dmd_predicts_up(
    prices: np.ndarray, horizon: int = 1, rank: int = 25, start_horizon: int = 0,
) -> bool:
    """Forecast direction between two future offsets from the last close."""
    prices = np.asarray(prices, dtype=float)
    n = len(prices)
    if (n < 4 or start_horizon < 0 or horizon <= start_horizon
            or not np.isfinite(prices).all() or (prices <= 0).any()):
        return False
    # Match the source DMD implementation, which fits raw prices.
    logp = prices.copy()
    row_len = n // 2 + 1
    hankel = np.column_stack([logp[i:i + row_len] for i in range(n - row_len + 1)])
    x, y = hankel[:, :-1], hankel[:, 1:]
    try:
        u, singular, vh = np.linalg.svd(x, full_matrices=False)
        r = min(rank, len(singular))
        if r == 0 or singular[r - 1] <= np.finfo(float).eps:
            return False
        ur, vr = u[:, :r], vh[:r, :].T
        inv_s = np.diag(1.0 / singular[:r])
        reduced = ur.T @ y @ vr @ inv_s
        eigenvalues, modes = np.linalg.eig(reduced)
        good = np.isfinite(eigenvalues) & (np.abs(eigenvalues) > 1e-12)
        if not good.any():
            return False
        eigenvalues, modes = eigenvalues[good], modes[:, good]
        phi = y @ vr @ inv_s @ modes
        amplitudes = np.linalg.lstsq(phi, hankel, rcond=None)[0][:, 0]
        last = phi[0] @ (amplitudes * eigenvalues ** (hankel.shape[1] - 1 + start_horizon))
        future = phi[0] @ (amplitudes * eigenvalues ** (hankel.shape[1] - 1 + horizon))
        return bool(np.isfinite(last) and np.isfinite(future) and future > last)
    except (np.linalg.LinAlgError, FloatingPointError):
        return False


def directional_score(prices: np.ndarray, windows: tuple[int, ...] = tuple(range(8, 25))):
    """Return the best-window DMD direction, using only a supplied trailing series."""
    prices = np.asarray(prices, dtype=float)
    usable = [n for n in windows if n <= len(prices)]
    if not usable:
        return np.nan
    # The original strategy selects windows by trailing hit rate. That online
    # selector is implemented in walk_forward.py; this is the raw forecast.
    return 1.0 if dmd_predicts_up(prices[-max(usable):]) else -1.0


def walk_forward_stock_signal(
    prices: np.ndarray,
    *,
    windows: tuple[int, ...] = tuple(range(3, 25)),
    selection_lookback: int = 60,
    invert: bool = True,
) -> np.ndarray:
    """Daily online hit-rate selection adapted from DMDBacktestV3.

    The forecast covers the close after execution (t+1) to the next close
    (t+2). Window selection uses only those outcomes known by signal date t.
    Output is 0 until enough history exists, otherwise
    +1/-1; the source script's ``INVERT_SIGNALS=True`` convention is retained.
    """
    p = np.asarray(prices, dtype=float)
    t_count = len(p)
    pred = np.full((len(windows), t_count), np.nan)
    for wi, size in enumerate(windows):
        for t in range(size - 1, t_count):
            segment = p[t-size+1:t+1]
            if np.isfinite(segment).all():
                pred[wi, t] = 1.0 if dmd_predicts_up(
                    segment, horizon=2, start_horizon=1,
                ) else -1.0
    out = np.zeros(t_count, dtype=np.float32)
    for t in range(t_count):
        # The latest fully observed t+1 to t+2 return was forecast at t-2.
        hi = t - 2
        lo = hi - selection_lookback + 1
        if lo < 0 or hi < lo:
            continue
        future = p[2:t+1] / p[1:t] - 1.0
        actual = np.sign(future)
        actual[~np.isfinite(future)] = 0
        scores = []
        for wi in range(len(windows)):
            hist = pred[wi, lo:hi+1]
            truth = actual[lo:hi+1]
            valid = np.isfinite(hist) & (truth != 0)
            scores.append(np.mean(np.sign(hist[valid]) == truth[valid]) if valid.sum() == selection_lookback else -1.0)
        best = int(np.argmax(scores))
        if scores[best] >= 0 and np.isfinite(pred[best, t]):
            out[t] = pred[best, t] * (-1 if invert else 1)
    return out

''')

## Implementation: inflation

In [10]:
inflation = register('inflation', r'''
"""Inflation-beta signal built from ALFRED vintage CPI and monthly CRSP returns.

Every CPI value must be selected from the vintage available on the signal
date. See README for the source and release-lag rules. Input arrays are dates
by stocks; all correlations are calculated from history strictly before the
signal date.
"""

from __future__ import annotations

import numpy as np
import polars as pl


def inflation_beta_signal(
    monthly: pl.DataFrame,
    *,
    return_column: str = "monthly_return",
    inflation_column: str = "inflation_change",
    window_months: int = 60,
    min_observations: int = 36,
) -> pl.DataFrame:
    """Return a dated stock score equal to negative rolling inflation beta.

    The score has higher values for lower inflation sensitivity, following the
    negative inflation risk price motivating Duarte (2013). Each asset's
    beta uses its own valid history; cross-sectional ranks are formed later on
    the daily investable universe.
    """
    required = {"date", "barrid", return_column, inflation_column}
    if missing := required - set(monthly.columns):
        raise ValueError(f"Missing columns: {sorted(missing)}")
    monthly = monthly.sort(["barrid", "date"])
    return (
        monthly.with_columns(
            pl.col(return_column).rolling_cov(
                pl.col(inflation_column), window_size=window_months,
                min_periods=min_observations,
            ).over("barrid").alias("cov_ri"),
            pl.col(inflation_column).rolling_var(
                window_size=window_months, min_periods=min_observations,
            ).over("barrid").alias("var_inflation"),
        )
        .with_columns((-pl.col("cov_ri") / pl.col("var_inflation")).alias("inflation_beta_alpha"))
        .select("date", "barrid", "inflation_beta_alpha")
    )


def market_beta_residual(
    daily: pl.DataFrame,
    *,
    return_column: str = "return",
    benchmark_return_column: str = "benchmark_return",
    beta_column: str = "predicted_beta",
) -> pl.DataFrame:
    """Make next-close targets using point-in-time market beta and benchmark."""
    required = {"date", "barrid", return_column, benchmark_return_column, beta_column}
    if missing := required - set(daily.columns):
        raise ValueError(f"Missing columns: {sorted(missing)}")
    return daily.with_columns(
        (pl.col(return_column) - pl.col(beta_column) * pl.col(benchmark_return_column))
        .shift(-1).over("barrid").alias("forward_residual_return"),
        pl.col("date").shift(-1).over("barrid").alias("label_available_date"),
    )


def _demeaned_rank(values: np.ndarray) -> np.ndarray:
    """Cross-sectional rank score with zero mean and unit standard deviation."""
    values = np.asarray(values, dtype=float)
    valid = np.isfinite(values)
    out = np.full(values.shape, np.nan)
    if valid.sum() < 2:
        return out
    ranks = np.argsort(np.argsort(values[valid])).astype(float)
    ranks = ranks / (len(ranks) - 1) - 0.5
    sd = ranks.std()
    out[valid] = (ranks - ranks.mean()) / (sd if sd > 0 else 1.0)
    return out

''')

## Implementation: macro

In [11]:
macro = register('macro', r'''
"""Yahoo market context and real-time ALFRED CPI snapshots.

Yahoo symbols match Jeffrey's Sharpe Transformer source (SPY, ^VIX, ^TNX,
^IRX). CPIAUCSL vintage CSVs are fetched from the St. Louis Fed graph endpoint
with ``vintage_date`` set to the final calendar day of each month. Thus each snapshot
contains only CPI observations available by that date; it can update a daily
signal after the CPI release without using later revisions.
"""

from __future__ import annotations

import calendar
import csv
import io
from datetime import date
from urllib.request import urlopen

import polars as pl


YAHOO_SYMBOLS = {"SPY": "spy", "^VIX": "vix", "^TNX": "tnx", "^IRX": "irx"}
ALFRED_URL = "https://alfred.stlouisfed.org/graph/alfredgraph.csv"


def fetch_yahoo_market(start: str, end: str) -> pl.DataFrame:
    """Download daily market context using the symbols in Jeffrey's notebook."""
    import yfinance as yf

    raw = yf.download(
        list(YAHOO_SYMBOLS), start=start, end=end, auto_adjust=False,
        actions=False, group_by="ticker", progress=False, threads=True,
    )
    if raw.empty:
        raise RuntimeError(f"Yahoo returned no observations for {start} through {end}")
    rows = []
    for symbol, alias in YAHOO_SYMBOLS.items():
        series = raw[symbol]["Close"] if isinstance(raw.columns, __import__("pandas").MultiIndex) else raw["Close"]
        rows.extend(
            {"date": idx.date(), "series": alias, "value": float(value)}
            for idx, value in series.dropna().items()
        )
    return pl.DataFrame(rows).pivot(on="series", index="date", values="value").sort("date")


def fetch_alfred_month_end_vintages(start: date, end: date) -> pl.DataFrame:
    """Download one point-in-time CPIAUCSL snapshot per month from ALFRED.

    ``vintage_date`` is month end, not the observation month. Consumers must
    join the latest vintage whose date is on or before the signal date.
    """
    rows = []
    year, month = start.year, start.month
    while (year, month) <= (end.year, end.month):
        vintage = date(year, month, calendar.monthrange(year, month)[1])
        url = f"{ALFRED_URL}?id=CPIAUCSL&vintage_date={vintage.isoformat()}"
        with urlopen(url, timeout=30) as response:
            csv_text = response.read().decode("utf-8")
        for row in csv.DictReader(io.StringIO(csv_text)):
            value = row[f"CPIAUCSL_{vintage:%Y%m%d}"]
            if value and value != ".":
                if date.fromisoformat(row["observation_date"]) >= vintage.replace(day=1):
                    raise ValueError(f"Unexpected observation in vintage {vintage}: {row}")
                rows.append({
                    "vintage_date": vintage,
                    "observation_date": date.fromisoformat(row["observation_date"]),
                    "cpi": float(value),
                })
        if month == 12:
            year, month = year + 1, 1
        else:
            month += 1
    return pl.DataFrame(rows).sort(["vintage_date", "observation_date"])


def vintage_inflation_changes(vintages: pl.DataFrame) -> pl.DataFrame:
    """Calculate the last available month-over-month change in YoY CPI per vintage."""
    required = {"vintage_date", "observation_date", "cpi"}
    if missing := required - set(vintages.columns):
        raise ValueError(f"Missing CPI vintage fields: {sorted(missing)}")
    vintage_rows = (
        vintages.sort(["vintage_date", "observation_date"])
        .with_columns(
            ((pl.col("cpi") / pl.col("cpi").shift(12).over("vintage_date") - 1) * 100)
            .alias("inflation_yoy")
        )
        .with_columns(
            (pl.col("inflation_yoy") - pl.col("inflation_yoy").shift(1).over("vintage_date"))
            .alias("inflation_change")
        )
        .drop_nulls("inflation_change")
        .group_by("vintage_date", maintain_order=True)
        .last()
        .select(pl.col("vintage_date").alias("date"), "inflation_change")
        .sort("date")
    )
    return vintage_rows

''')

## Configuration and prior
The penalty is λ‖β − c·1‖² on standardized signals with fixed c = 10⁻⁵ on every date. Both signals share this center. The fitted coefficients can still differ. Equal-weight means equal weights on standardized signal scores. Zero beta is **active beta**, so total beta equals benchmark beta; total weights sum to one. BMA evidence comes only from unrestricted active portfolios; the same resulting model weights are used to form the long-only BMA portfolio.

In [12]:
PENALTIES = {63: 18, 252: 5, 1260: 2.5}
FIXED_PRIOR = 1e-5
TE_CAP = 0.01
END = __import__('datetime').date(2024,12,31)
portfolio.TARGET_TRACKING_ERROR = TE_CAP
SIGNALS = ('dmd','inflation_beta')


## Export the actual analysis dataset and risk snapshots
This export uses corrected, previously generated point-in-time DMD and inflation forecasts. It exports the full benchmark universe, matured returns, and dated Barra styles. Missing style coverage is tracked, with unmatched DMD assigned zero. Do not distribute licensed underlying data beyond permitted recipients. Historical result files alone are insufficient to rerun the optimizer.

In [13]:
def prepare_dataset(cache=Path.home()/'nobackup/autodelete/botq2026/corrected-signals'):
    import sf_quant.data as sfd
    sfd.env(root=str(Path.home()), database='production')
    assets=pl.scan_parquet(cache/'assets.parquet').filter(pl.col('date').is_between(__import__('datetime').date(2005,1,1),END)).select('date','barrid','in_universe','market_cap','return','predicted_beta').collect()
    forecasts=pl.read_parquet(cache/'forecasts-through-2024.parquet').select('signal_date','barrid',*SIGNALS)
    panel=walk_forward.build_panel(assets,forecasts,pl.read_parquet(cache/'exit_return_audit.parquet'))
    parts=[]
    for year in panel.with_columns(pl.col('signal_date').dt.year().alias('_year')).partition_by('_year',include_key=False):
        styles=sfd.load_exposures(year['signal_date'].min(),year['signal_date'].max(),False,['USSLOWL_MOMENTUM','USSLOWL_LTREVRSL'])
        parts.append(year.join(styles.rename({'date':'signal_date'}),on=['signal_date','barrid'],how='left',validate='m:1'))
    panel=pl.concat(parts).sort('date','barrid')
    panel.write_parquet(DATA/'analysis_panel.parquet')
    print('Analysis panel saved:',panel.shape,flush=True)
    # Years are independent; limit concurrent archives to the Slurm CPU allocation.
    from concurrent.futures import ThreadPoolExecutor
    import os
    years=panel.filter(pl.col('date').dt.year()>=2010).with_columns(pl.col('date').dt.year().alias('_year')).partition_by('_year',include_key=False)
    def risk_year(year):
        destination=DATA/f'risk_{year["date"][0].year}.npz'
        if destination.exists(): return
        arrays={}
        for day in year.partition_by('date',maintain_order=True):
            key=str(day['date'][0]);ids=day['barrid'].to_list()
            e,f,s=sfd.construct_factor_model_components(day['date'][0],ids)
            arrays[key+'_ids']=np.asarray(ids,dtype=str)
            arrays[key+'_E']=e;arrays[key+'_F']=f;arrays[key+'_S']=s
        np.savez_compressed(destination,**arrays)
        print('Saved risk archive',destination,flush=True)
    with ThreadPoolExecutor(max_workers=min(4,int(os.environ.get('SLURM_CPUS_PER_TASK','4')))) as pool:
        list(pool.map(risk_year,years))
    return panel
# Run prepare_dataset() on the cluster to populate the final dataset.


## Final backtest
Fits use only labels available strictly before execution. BMA updates from realized candidate active returns available strictly before execution. BMA uses the existing Gaussian evidence rule (1% annual alternative mean, 2.5% reference volatility); this is a specified evidence model, not proof that a candidate is the true model. Candidate scores are normalized before blending. Both variants are optimized at the new 1% cap, including standalone signals and matched original DMD.

In [14]:
def run_final(baselines_only=False):
    if not baselines_only:
        assert all(v is not None and v>=0 for v in PENALTIES.values()), 'Set final penalties first'
    panel=pl.read_parquet(DATA/'analysis_panel.parquet').sort('date','barrid')
    diagnostics=[];parts=[]
    for day in panel.partition_by('date',maintain_order=True):
        raw=day['dmd'].to_numpy(); styles=day.select('USSLOWL_MOMENTUM','USSLOWL_LTREVRSL').to_numpy()
        mask=np.isfinite(raw)&np.isfinite(styles).all(axis=1)
        if mask.sum()<30: raise ValueError('Insufficient style coverage')
        x=styles[mask]-styles[mask].mean(axis=0); y=raw[mask]-raw[mask].mean()
        residual=y-x@np.linalg.lstsq(x,y,rcond=None)[0]
        clean=np.zeros(len(raw));matched=np.zeros(len(raw));clean[mask]=residual;matched[mask]=y
        diagnostics.append({'date':day['date'][0],'coverage':mask.mean(),'r2':1-residual@residual/(y@y) if y@y else np.nan})
        parts.append(day.with_columns(pl.Series('dmd',clean),pl.Series('raw_dmd',matched)))
    panel=pl.concat(parts)
    pl.DataFrame(diagnostics).write_parquet(DATA/'style_diagnostics.parquet')
    inputs=combination.daily_ridge_inputs(panel,signals=SIGNALS)
    days=panel.filter((pl.col('date').dt.year()>=2010)&(pl.col('label_available_date')<=END)).partition_by('date',maintain_order=True)
    dates=[d['date'][0] for d in days]
    fits={} if baselines_only else {dt:{f'ridge_{half}_p{penalty:g}':(combination.fit_daily_ridge(inputs,as_of=dt,half_life_days=half,penalty=penalty,center=FIXED_PRIOR),FIXED_PRIOR) for half,penalty in PENALTIES.items()} for dt in dates}
    import threading, os
    from concurrent.futures import ThreadPoolExecutor
    rows=[];coefs=[];factor_rows=[];worker_state=threading.local()
    workers=min(8,int(os.environ.get('SLURM_CPUS_PER_TASK','4')))
    def evaluate(day, name, alpha, long_only):
        dt=day['date'][0];key=str(dt)
        if getattr(worker_state,'risk_year',None)!=dt.year:
            if getattr(worker_state,'archive',None) is not None: worker_state.archive.close()
            worker_state.archive=np.load(DATA/f'risk_{dt.year}.npz',allow_pickle=False)
            worker_state.risk_year=dt.year
        archive=worker_state.archive
        assert list(archive[key+'_ids'])==day['barrid'].to_list()
        E,F,S=[archive[key+'_'+s] for s in ('E','F','S')]
        b=day['benchmark_weight'].to_numpy();beta=day['predicted_beta'].to_numpy()
        a,te=sparse_optimizer.solve(alpha/max(np.max(np.abs(alpha)),1e-20),E,F,S,b,beta,long_only=long_only,risk_cap=TE_CAP)
        total=b+a; ret=day['holding_return'].to_numpy()
        assert abs(total.sum()-1)<1e-5 and abs(a@beta)<1e-5 and te<=TE_CAP+1e-6
        if long_only: assert total.min()>-1e-6
        variant='long_only' if long_only else 'active';label=name+'_'+variant
        exposures=E.T@a
        factor_rows.extend({'date':dt,'portfolio':label,'factor_index':j,'active_exposure':float(v),'variance_contribution':float(v*(F@exposures)[j])} for j,v in enumerate(exposures))
        return dict(date=day['label_available_date'][0],portfolio=label,active_return=float(a@ret),portfolio_return=float(total@ret),benchmark_return=float(b@ret),predicted_te=te,active_beta=float(a@beta),total_investment=float(total.sum()),minimum_total_weight=float(total.min()),specific_variance=float((a*a)@S),lagged_momentum_exposure=float(a@np.nan_to_num(day['USSLOWL_MOMENTUM'].to_numpy())),lagged_reversal_exposure=float(a@np.nan_to_num(day['USSLOWL_LTREVRSL'].to_numpy())))
    def alphas(day):
        dt=day['date'][0];scores=day.select(SIGNALS).to_numpy();out={}
        for name,(fit,center) in fits.get(dt,{}).items():
            z=np.nan_to_num((scores-fit.means)/fit.scales)
            out[name]=z@fit.coefficients
        f=combination.fit_daily_ridge(inputs,as_of=dt,half_life_days=126,penalty=1)
        z=np.nan_to_num((scores-f.means)/f.scales)
        out['equal_weight']=z.mean(axis=1)
        return out
    def candidate_day(day):
        dt=day['date'][0];candidates=alphas(day)
        coefficients=[dict(date=dt,method=name,center=center,**dict(zip(SIGNALS,fit.coefficients))) for name,(fit,center) in fits.get(dt,{}).items()]
        candidates.update(dmd_standalone=day['dmd'].to_numpy(),inflation_standalone=np.nan_to_num(day['inflation_beta'].to_numpy()),original_dmd_matched=day['raw_dmd'].to_numpy())
        result=[]
        for name,alpha in candidates.items():
            variants=(False,True) if name in ('dmd_standalone','inflation_standalone','original_dmd_matched') else (False,)
            for lo in variants: result.append(evaluate(day,name,alpha,lo))
        return result,coefficients
    with ThreadPoolExecutor(max_workers=workers) as pool:
        for index,(result,coefficients) in enumerate(pool.map(candidate_day,days),1):
            rows.extend(result);coefs.extend(coefficients)
            if index%250==0: print('Backtest',index,'/',len(days),flush=True)
    history=pl.DataFrame(rows)
    if baselines_only:
        history.write_parquet(DATA/'baseline_returns.parquet')
        pl.DataFrame(factor_rows).write_parquet(DATA/'baseline_factor_exposures.parquet')
        print('Penalty-independent backtests complete',flush=True)
        return
    names=list(fits[dates[0]])+['equal_weight']
    from scipy.special import softmax
    columns=[name+'_active' for name in names]
    active_history=history.filter(pl.col('portfolio').is_in(columns)).pivot(on='portfolio',index='date',values='active_return').sort('date')
    assert set(active_history.columns)=={'date',*columns} and active_history.height==len(dates)
    realized=active_history.select(columns).to_numpy()
    assert np.isfinite(realized).all()
    known=np.searchsorted(active_history['date'].to_numpy(),np.asarray(dates,dtype='datetime64[D]'),side='left')
    cumulative=np.vstack([np.zeros(len(names)),realized.cumsum(axis=0)])
    active_post=dict(zip(dates,softmax((0.01/0.025**2)*cumulative[known],axis=1)))
    post={'active':active_post,'long_only':active_post}
    def bayes_day(day):
        dt=day['date'][0];a=alphas(day);matrix=np.column_stack([a[n] for n in names]);sd=matrix.std(axis=0)
        matrix=np.divide(matrix,sd,out=np.zeros_like(matrix),where=sd>0)
        return [evaluate(day,'bayesian',matrix@post['long_only' if lo else 'active'][dt],lo) for lo in (False,True)]
    with ThreadPoolExecutor(max_workers=workers) as pool:
        for index,result in enumerate(pool.map(bayes_day,days),1):
            rows.extend(result)
            if index%250==0: print('Bayesian backtest',index,'/',len(days),flush=True)
    pl.DataFrame(rows).write_parquet(DATA/'final_returns.parquet')
    pl.DataFrame(coefs).write_parquet(DATA/'final_coefficients.parquet')
    pl.DataFrame(factor_rows).write_parquet(DATA/'final_factor_exposures.parquet')
    pl.DataFrame([dict(date=dt,variant=v,**dict(zip(names,w))) for v,series in post.items() for dt,w in series.items()]).write_parquet(DATA/'final_model_weights.parquet')
# run_final() after the dataset export.
